# Aether Stage 2 — smoke test

Run this notebook before the training notebook. It validates the software stack on a small sample: unit tests, private Stage 1 checkpoint loading, real Mimi and Qwen paths, cached SLUE-SQA-5 preparation, gradient flow, generation, checkpointing, and a bounded tiny-overfit run.

Artifacts are written to a new timestamped directory under `MyDrive/aether-v2/stage2/`. The Hugging Face token is read from the Colab secret named `HF_TOKEN`; it is never printed or written into the notebook.

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, subprocess, sys
from pathlib import Path
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "huggingface_hub", "pytest", "ruff", "mypy"], check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Repository tests

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 2. Load the private Stage 1 encoder

In [ ]:
CONFIG_PATH = "configs/stage2_smoke.yaml"
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from aether_v3.config import load_config
from aether_v3.models.aether_speech import AetherSpeechEncoder
from aether_v3.training.stage2_utils import load_stage1_encoder

cfg = load_config(CONFIG_PATH)
stage1_path = hf_hub_download(
    repo_id=cfg.stage2_train.stage1_repo_id,
    filename=cfg.stage2_train.stage1_filename,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
encoder = AetherSpeechEncoder(cfg.aether_speech)
checkpoint = load_stage1_encoder(stage1_path, encoder)
encoder.eval()
print("Stage 1 encoder loaded; checkpoint step:", checkpoint.get("step", "unknown"))

## 3. Build a tiny real SLUE-SQA-5 cache

This uses spoken question audio and the linked document as text. The question transcript is retained by the source dataset for diagnostics but is never passed to Qwen.

In [ ]:
from datasets import load_dataset
from aether_v3.data.stage2_cache import build_slue_sqa5_shards
from aether_v3.models.mimi_wrapper import FrozenMimi
from aether_v3.training.stage2_utils import create_run_dir

run_dir = create_run_dir(cfg.stage2_train.drive_root)
cache_root = run_dir / "cache"
train_rows = load_dataset(cfg.stage2_data.dataset_id, cfg.stage2_data.dataset_config, split="train", streaming=True, token=os.environ["HF_TOKEN"], trust_remote_code=True)
val_rows = load_dataset(cfg.stage2_data.dataset_id, cfg.stage2_data.dataset_config, split="validation", streaming=True, token=os.environ["HF_TOKEN"], trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained(cfg.llm.model_id, revision=cfg.llm.revision)
mimi = FrozenMimi(cfg.mimi.pretrained_id, cfg.mimi.num_quantizers, device="cuda")
build_slue_sqa5_shards(train_rows, mimi, encoder, tokenizer, cache_root/"train", "train", max_examples=64)
build_slue_sqa5_shards(val_rows, mimi, encoder, tokenizer, cache_root/"validation", "validation", max_examples=16)
print("run_dir:", run_dir)

## 4. Real forward, gradient, generation, and tiny overfit

In [ ]:
from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.training.train_stage2 import run_stage2_training

model = AetherSpeechLLM(cfg.aether_speech, cfg.connector, cfg.llm, speech_frozen=True)
load_stage1_encoder(stage1_path, model.encoder)
run_stage2_training(cfg, run_dir, cache_root/"train", cache_root/"validation", model=model)
assert (run_dir/"last.pt").exists()
assert (run_dir/"best_val_loss.pt").exists()
assert (run_dir/"best_answer_f1.pt").exists()
print("SMOKE TEST PASSED:", run_dir)

## Pass criteria

The notebook passes only if all cells finish, loss is finite, Connector gradients flow, generated answers are produced, and `last.pt`, periodic checkpoints, and best checkpoints exist on Drive. A decreasing tiny-overfit loss is the gate before starting the full notebook.